In [ ]:
from time import sleep as delay
from project.el5600 import project
if "ic" in globals():
    ic.close()
ic = project(device="el5600", revision="a1", emulator="cp2112", logging=False, is_gui=False)

from phy.multimeter import keithley_dm6500, rohde_hmc8012, dmm
from phy.power_supply import keithley_2470, keysight_e36232a
from phy.scope import tektronix_dp_series_usb
from phy.eloader import sdl1030x
# from phy.relay_16ch import relay_box
# from phy.chamber_th3 import chamber

%matplotlib tk
from interface.docs.output_chart import plot
from interface.cui_colors import color
from interface.i2c_bridge.tca9548a import tca9548
from interface.docs.output_excel import excel_frame, style
from interface.cui_logger import logger as log

import numpy as np

chart = plot()

dm1 = keithley_dm6500()
dm2 = rohde_hmc8012()
ps = keysight_e36232a()
sm = keithley_2470()
ld = sdl1030x()
ds = tektronix_dp_series_usb()

# relay = relay_box(i2c_h=ic)
# tc = chamber(port=3)
# mux = tca9548(ic.i2c_h, 0x70)

# --------------------------------------------------
# list_voltage = list(np.arange(5, 8, 0.005))
# voltage  = [round(num, 3) for num in list_voltage]
# --------------------------------------------------

In [ ]:
# connection
# E36232 --- DM2 HMC8012 --- VIN
# VOUT --- SDL1030
# DM1 DMM6500 --- INTB (isns GPIO)
# EN --- 3.3V (thru N-MOS, gate = GPIO 7)

In [ ]:
def disable_all():
    ld.disable
    sm.disable
    ps.disable
    
    ld.vset = 5
    sm.vset = 3.3
    ps.vset = 5.0

In [ ]:
disable_all()

h = ic.get_i2c_handler()
h.gpio_7(0) # EN pin control

ld.disable
ps.disable
sm.cfg_all = 3.3, 0.1
sm.power_recycle

initial_isgn = ic.trm_isgn
initial_isofs = ic.trm_isofs

print(f"initial_isgn  : {initial_isgn}")
print(f"initial_isofs : {initial_isofs}")

# disconnect EN
# activate tst mode
ic.write_byte(0xa0, 0xf9)
ic.write_byte(0xa0, 0x9f)
print(f"Read byte : [0xa0]={ic.read_byte(0xa0):#04x}")

In [ ]:
# pre-load
ic.write_byte(0xc0, 0x37)
ic.write_byte(0xc4, 0x01)
ic.write_byte(0xc5, 0x38)

ic.write_byte(0xa1, 0x31)
ic.write_byte(0xa1, 0x00)

ic.write_byte(0xa2, 0x31)
ic.write_byte(0xa2, 0x00)

ic.write_byte(0xa1, 0x62)
ic.write_byte(0xa7, 0x80)
ic.write_byte(0xa7, 0xc0)
ic.write_byte(0xa3, 0x10)
ic.write_byte(0xa1, 0x21)
ic.write_byte(0xa1, 0x22)
ic.write_byte(0xa3, 0x18)

ic.write_byte(0xbd, 0x04)
ic.write_byte(0xa3, 0x00)
ic.write_byte(0xa3, 0x04)
ic.write_byte(0xa3, 0x00)

ic.write_byte(0xa1, 0x32)
ic.write_byte(0xa1, 0x21)
ic.write_byte(0Xa1, 0x52)

ic.write_byte(0xa1, 0x00)
ic.write_byte(0xa7, 0x00)

ps.vset = 5
ps.power_recycle

ic.write_byte(0xbd, 0x00)
ic.write_byte(0xa2, 0x44)
ic.write_byte(0x0c, 0x40)

ic.write_byte(0xa2, 0x04)

In [ ]:
h.gpio_7(1)

In [ ]:
# trm_isgn[5:0]
# trm_isofs[2:0]

ps.cfg_all = 20, 5

ic.write_byte(0xa6, 0x10)
ic.write_byte(0xc5, 0xa8)
ic.write_byte(0xa2, 0x24)
ic.write_byte(0xca, 0x9f)
ic.trm_isofs = 4

In [ ]:
# calculate R_cable
ld.disable
delay(1)
ld.vset = 20
ld.enable
delay(1)

# ic.ILIM_TH = 31
# ic.ILIM_TH_EX = 1

ld.vset = 19.5
delay(1)
iload_19p5 = dm2.current_10A
r_cable = 0.5 / iload_19p5
print(f"R_cable : {r_cable:.06f}")

ld.vset = 20
delay(1)

In [ ]:
# --------------------------------------------------
evb_no = "evb#1 (v_mode, iout measure, add cap)"
# --------------------------------------------------

from datetime import datetime
date_str = datetime.now().strftime("%Y%m%d")

# change the current scale to 10A
dm2.current_10A

ld.vset = 20
ld.power_recycle

log.output_set_filename(f"[{date_str}_EL6200_A1] trm_isgn - {evb_no}")

log.output_csv(["trm_isgn", "trm_isofs"])
log.output_csv([initial_isgn, initial_isofs])

print(f"[initial value] trm_isgn  : {initial_isgn}")
print(f"[initial value] trm_isofs : {initial_isofs}")

ic.trm_isgn = 0
ic.trm_isofs = 0

print(f"reset trm_isgn  : {initial_isgn:#x} --> {ic.trm_isgn:#x}")
print(f"reset trm_isofs : {initial_isofs:#x} --> {ic.trm_isofs:#x}")

In [ ]:
for trim_bit in range(1):
    
    ic.trm_isgn = trim_bit
    ic.write_byte(0x0c, 0x40)
    
    start_voltage = 20 - (1 * r_cable) # 1A

    list_item = list(np.arange(17.5, start_voltage, 0.005))
    for_item  = [round(num, 3) for num in list_item]
    for_item.reverse()
    
    coarse = 0
    fist_op = True
    print("find coarse value (lo)")

    for item_v in for_item:
        
        ld.vset = item_v
        if fist_op:
            delay(0.5)
            fist_op = False
        
        read_i = abs(round(dm2.current_10A, 6))
        read_cocp = abs(round(dm1.voltage_10, 1))
        
        isofs = ic.trm_isofs
        print(f"[{trim_bit:#04x}, {isofs:#04x}] {read_i}, {read_cocp}")
        
        if read_cocp > 0.5:
            ld.vset = 20
            break
        else:
            coarse = round(item_v + 0.07, 6)

    ld.vset = 20
    isofs = ic.trm_isofs
    print(f"[{trim_bit:#04x}, {isofs:#04x}] coarse voltage : {coarse}")
    delay(1)
    
    # --------------------------------------------------

    list_item = list(np.arange(17.5, coarse, 0.003))
    for_item  = [round(num, 3) for num in list_item]
    for_item.reverse()
    
    pre_isns_lo = 0
    fist_op = True
    print("find pre_isns_lo value")
    delay(1)

    for item_v in for_item:

        ld.vset = item_v
        if fist_op:
            delay(0.5)
            fist_op = False
        
        read_i = abs(round(dm2.current_10A, 6))
        read_cocp = abs(round(dm1.voltage_10, 1))
        
        isofs = ic.trm_isofs
        print(f"[{trim_bit:#04x}, {isofs:#04x}] {read_i}, {read_cocp}")
        
        if read_cocp > 1.5:
            ld.vset = 20
            break
        else:
            pre_isns_lo = round(read_i, 6)
        
    ld.vset = 20
    isofs = ic.trm_isofs
    print(f"[{trim_bit:#04x}, {isofs:#04x}] pre_isns_lo voltage : {pre_isns_lo}")
    
    # --------------------------------------------------

    previous_0Ch = ic.read_byte(0x0c)
    print(f"0x0c : {previous_0Ch:#04x}")
    
    ic.write_byte(0x0c, 0x5f)
    print(f"0x0c : {ic.read_byte(0x0c):#04x}")

    start_voltage = 20 - (1.7 * r_cable) # 1.7A

    list_item = list(np.arange(17.5, start_voltage, 0.005))
    for_item  = [round(num, 2) for num in list_item]
    for_item.reverse()

    coarse = 0
    fist_op = True
    print("find coarse value (hi)")
    delay(1)

    for item_v in for_item:

        ld.vset = item_v
        if fist_op:
            delay(0.5)
            fist_op = False
        
        read_i = abs(round(dm2.current_10A, 6))
        read_cocp = abs(round(dm1.voltage_10, 1))
        
        isofs = ic.trm_isofs
        print(f"[{trim_bit:#04x}, {isofs:#04x}] {read_i}, {read_cocp}")
        
        if read_cocp > 1.5:
            ld.vset = 20
            break
        else:
            coarse = round(item_v + 0.17, 6)

    ld.vset = 20
    isofs = ic.trm_isofs
    print(f"[{trim_bit:#04x}, {isofs:#04x}] coarse voltage : {coarse}")
    
    # --------------------------------------------------

    list_item = list(np.arange(17.5, coarse, 0.003))
    for_item  = [round(num, 3) for num in list_item]
    for_item.reverse()

    pre_isns_hi = 0
    fist_op = True
    print("find pre_isns_hi value")
    delay(1)    
    
    for item_v in for_item:

        ld.vset = item_v
        if fist_op:
            delay(0.5)
            fist_op = False
        
        read_i = round(dm2.current_10A, 6)
        read_cocp = abs(round(dm1.voltage_10, 1))
        
        isofs = ic.trm_isofs
        print(f"[{trim_bit:#04x}, {isofs:#04x}] {read_i}, {read_cocp}")
        
        if read_cocp > 0.5:
            ld.vset = 20
            break
        else:
            pre_isns_hi = round(read_i, 6)

    ld.vset = 20
    isofs = ic.trm_isofs
    print(f"[{trim_bit:#04x}, {isofs:#04x}] pre_isns_hi voltage : {pre_isns_hi}")

    isns_delta = round((pre_isns_hi - pre_isns_lo), 6)

    print(f"[isgn={trim_bit:#04x}, isofs={ic.trm_isofs}] isns_delta (target {round(1.003*1.85, 5)}) : {isns_delta:.05f}")
    
    if isns_delta > 1.9:
        break

In [ ]:
# adjust the start bit for trim
# check the inital isns_delta
# 0.1 isns_delta / 10 trim code
# e.g. : if code 0x00 and delta is 1.5, code 25d will be delta 1.75

start_fine_trim = 27

In [ ]:
log.output_csv([])
log.output_csv([])
log.output_csv([])
log.output_csv(["trm_isgn", "trm_isofs", "isns_lo", "isns_hi", "isns_delta"])

for trim_bit in range(start_fine_trim, start_fine_trim + 14):
    
    ic.trm_isgn = trim_bit
    ic.write_byte(0x0c, 0x40)
    
    start_voltage = 20 - (0.7 * r_cable) # 0.8A

    list_item = list(np.arange(17.5, start_voltage, 0.008))
    for_item  = [round(num, 3) for num in list_item]
    for_item.reverse()
    
    coarse = 0
    fist_op = True
    
    print("find coarse value (lo)")

    for item_v in for_item:
        
        ld.vset = item_v
        if fist_op:
            delay(0.5)
            fist_op = False
        
        read_i = abs(round(dm2.current_10A, 6))
        read_cocp = abs(round(dm1.voltage_10, 1))
        
        isofs = ic.trm_isofs
        # print(f"[{trim_bit:#04x}, {isofs:#04x}] {read_i}, {read_cocp}")
        
        if read_cocp > 1:
            ld.vset = 20
            break
        else:
            offset_v = r_cable * 0.2
            coarse = round(item_v + offset_v, 6)

    ld.vset = 20
    isofs = ic.trm_isofs
    print(f"[{trim_bit:#04x}, {isofs:#04x}] coarse voltage : {coarse}")
    delay(1)
    
    # --------------------------------------------------

    # list_item = list(np.arange(17.5, coarse, 0.002))
    list_item = list(np.arange(17.5, coarse, 0.001))
    
    for_item  = [round(num, 3) for num in list_item]
    for_item.reverse()
    
    pre_isns_lo = 0
    fist_op = True
    print("find pre_isns_lo value")
    delay(1)
    
    count = 0

    for item_v in for_item:

        ld.vset = item_v
        if fist_op:
            delay(0.5)
            fist_op = False
        
        read_i = abs(round(dm2.current_10A, 6))
        read_cocp = abs(round(dm1.voltage_10, 1))
        
        isofs = ic.trm_isofs
        
        # print(f"[{trim_bit:#04x}, {isofs:#04x}] {read_i}, {read_cocp}")
        print(trim_bit, isofs, read_i, read_cocp, ic.SW_STS, ic.OFF_BY_ILIM, ic.OC_STS, count)
        
        if read_cocp > 1:
            count += 1
        else:
            pre_isns_lo = round(read_i, 6)
        
        if count > 10:
            break
        
    ld.vset = 20
    isofs = ic.trm_isofs
    print(f"[{trim_bit:#04x}, {isofs:#04x}] pre_isns_lo voltage : {pre_isns_lo}")
    
    # --------------------------------------------------

    previous_0Ch = ic.read_byte(0x0c)
    # print(f"0x0c : {previous_0Ch:#04x}")
    
    ic.write_byte(0x0c, 0x5f)
    # print(f"0x0c : {ic.read_byte(0x0c):#04x}")

    start_voltage = 20 - (2.5 * r_cable) # 2.5A

    list_item = list(np.arange(17.5, start_voltage, 0.008))
    for_item  = [round(num, 2) for num in list_item]
    for_item.reverse()

    coarse = 0
    fist_op = True
    print("find coarse value (hi)")
    delay(1)

    for item_v in for_item:

        ld.vset = item_v
        if fist_op:
            delay(0.5)
            fist_op = False
        
        read_i = abs(round(dm2.current_10A, 6))
        for _ in range(2):
            read_cocp = abs(round(dm1.voltage_10, 1))
        
        isofs = ic.trm_isofs
        # print(f"[{trim_bit:#04x}, {isofs:#04x}] {read_i}, {read_cocp}")
        
        if read_cocp > 1:
            ld.vset = 20
            break
        else:
            offset_v = r_cable * 0.2
            coarse = round(item_v + offset_v, 6)

    ld.vset = 20
    isofs = ic.trm_isofs
    print(f"[{trim_bit:#04x}, {isofs:#04x}] coarse voltage : {coarse}")
    
    # --------------------------------------------------

    # list_item = list(np.arange(17.5, coarse, 0.002))
    list_item = list(np.arange(17.5, coarse, 0.001))
    for_item  = [round(num, 3) for num in list_item]
    for_item.reverse()

    pre_isns_hi = 0
    fist_op = True
    print("find pre_isns_hi value")
    delay(1)
    
    count = 0
    
    for item_v in for_item:

        ld.vset = item_v
        if fist_op:
            delay(0.5)
            fist_op = False
        
        read_i = round(dm2.current_10A, 6)
        for _ in range(2):
            read_cocp = abs(round(dm1.voltage_10, 1))
        
        isofs = ic.trm_isofs
        
        # print(f"[{trim_bit:#04x}, {isofs:#04x}] {read_i}, {read_cocp}")
        print(trim_bit, isofs, read_i, read_cocp, ic.SW_STS, ic.OFF_BY_ILIM, ic.OC_STS, count)
        
        if read_cocp > 1:
            count += 1
        else:
            pre_isns_hi = round(read_i, 6)
        
        if count > 10:
            break

    ld.vset = 20
    delay(1)   
    
    isofs = ic.trm_isofs
    print(f"[{trim_bit:#04x}, {isofs:#04x}] pre_isns_hi voltage : {pre_isns_hi}")

    isns_delta = round((pre_isns_hi - pre_isns_lo), 6)

    isofs = ic.trm_isofs
    log.output_csv([trim_bit, isofs, pre_isns_lo, pre_isns_hi, isns_delta])
    print(f"[isgn={trim_bit:#04x}, isofs={ic.trm_isofs}] isns_delta (target {round(1.003*1.85, 5)}) : {isns_delta:.05f}")

In [ ]:
# overwrite the trm_isgn
# target : 1.85555

# --------------------------------------------------
ic.trm_isgn = 37
ic.trm_isofs = 0
# --------------------------------------------------

ic.write_byte(0x0c, 0x40)
ic.otp_lorng = 0

print(f"trm_isgn : {ic.trm_isgn}")
print(f"trm_isofs : {ic.trm_isofs}")
print(f"otp_lorng (0b) : {ic.otp_lorng}")
print(f"0x0c (0x40) : {ic.read_byte(0x0c):#04x}")
print(f"otp_lorng (0b) : {ic.otp_lorng}")

In [ ]:
log.output_csv([])
log.output_csv([])
log.output_csv([])
log.output_csv(["trm_isgn", "trm_isofs", "transition current (A)"])

# target start point : 3A
start_vin = 20 - round(2.9 * r_cable, 3)

list_item = list(np.arange(18, start_vin, 0.002))
for_item  = [round(num, 3) for num in list_item]
for_item.reverse()

pre_iset = 0

for trim_bit in range(0b1_000):
    
    ic.trm_isofs = trim_bit
    fist_op = True

    for item_v in for_item:

        ld.vset = item_v
        if fist_op:
            delay(0.5)
            fist_op = False
        
        read_i = abs(round(dm2.current_10A, 6))
        read_cocp = abs(round(dm1.voltage_10, 1))
        
        isgn = ic.trm_isgn
        print(f"[{isgn:#04x}, {trim_bit:#04x}] {read_i}, {read_cocp}")
        
        if read_cocp > 0.5:
            ld.vset = 20
            break
        else:
            pre_iset = round(read_i, 6)

    ld.vset = 20
    
    isgn = ic.trm_isgn
    print(f"[{isgn:#04x}, {trim_bit:#04x}] fine current : {pre_iset}")
    delay(1)
    
    log.output_csv([isgn, trim_bit, pre_iset])

# target : 3.25 + 0.022 = 3.272

In [ ]:
disable_all()
bench_isgn = 37
bench_isofs = 4

h.gpio_7(0)
delay(0.5)

In [ ]:
# calculate R_cable
h.gpio_7(0)
sm.power_recycle
ps.vset = 20
ps.power_recycle
ld.disable
delay(1)

ld.vset = 20
delay(0.5)

ic.ILIM_TH = 31
ic.ILIM_TH_EX = 1

h.gpio_7(1)
delay(0.5)

ld.vset = 19.5
ld.enable
delay(1)

iload_19p5 = dm2.current_10A
r_cable = 0.5 / iload_19p5
print(f"R_cable : {r_cable:.06f}")

ld.vset = 20
delay(1)

disable_all()

In [ ]:
# --------------------------------------------------
# ILIM_RANGE verification
# --------------------------------------------------

h = ic.get_i2c_handler()

# --------------------------------------------------
evb_no = "evb#1 (v_mode, iout measure, add cap)"
# --------------------------------------------------

from datetime import datetime
date_str = datetime.now().strftime("%Y%m%d")

log.output_set_filename(f"[{date_str}_EL6200_A1] trm_isgn - {evb_no}")

dm2.current_10A
ld.vset = 20
# --------------------------------------------------

log.output_csv([])
log.output_csv([])
log.output_csv([])
log.output_csv(["ILIM_RANGE verification - re-add cap"])

sm.cfg_all = 3.3, 0.1
sm.power_recycle
ps.cfg_all = 12, 6.5
ps.power_recycle

print(f"initial trm_isgn : {ic.trm_isgn:#x} ({ic.trm_isgn})")
print(f"initial trm_isofs : {ic.trm_isofs:#x}")

# activate tst mode
ic.write_byte(0xa0, 0xf9)
ic.write_byte(0xa0, 0x9f)
print(f"Read byte : [0xa0]={ic.read_byte(0xa0):#04x}")

# --------------------------------------------------
ic.trm_isgn = bench_isgn
ic.trm_isofs = bench_isofs
print(f"retrimmed trm_isgn : {ic.trm_isgn:#x} ({ic.trm_isgn})")
print(f"retrimmed trm_isofs : {ic.trm_isofs:#x}")
# --------------------------------------------------

ic.ILIM_HYS = 0 # 0mA
ic.ILIM_TH = 0
ic.ILIM_TH_EX = 0
ic.ACTIVE_ILIM_EN = 0 # latch
ic.OC_FAULT_PIN_EN = 1

In [ ]:
log.output_csv(["Output current measurement"])

In [ ]:
log.output_csv(["trm_isgn", "trm_isofs", "VIN (V)", "I_LOAD (A)", "ILIM_TH_EX", "ACTIVE_ILIM_EN", "Accuracy (%)"])

In [ ]:
ic.ILIM_BLANK_TIME = 0

In [ ]:
# vin = [20, 28]
vin = [28]

# current : [ILIM_TH, ILIM_TH_EX]
dict_load = {
    # 3.3 : [2, 0],
    # 4.1 : [2, 1],
    5   : [27, 0],
    # 5.5 : [15, 1],
    # 5.9 : [31, 1]
}

dm2.current_10A

for voltage in vin:
    
    ps.cfg_all = voltage, 6.5
    delay(1)

    count = 0
    
    ld.vset = voltage
    ld.enable
    delay(1)
    
    # activate tst mode|
    ic.write_byte(0xa0, 0xf9)
    ic.write_byte(0xa0, 0x9f)
    print(f"Read byte : [0xa0]={ic.read_byte(0xa0):#04x}")
    
    ic.trm_isgn = bench_isgn
    ic.trm_isofs = bench_isofs
    print(f"trm_isgn={ic.trm_isgn:#x} ({ic.trm_isgn}), trm_isofs={ic.trm_isofs:#x}")

    for current, reg in dict_load.items():
        
        h.gpio_7(1)
        delay(1)
        
        v1 = voltage - (r_cable * current * 0.7)
        v2 = voltage - (r_cable * current * 1.1)
        
        step = 0.001
        ndigit = 3

        list_temp = list(np.arange(v2, v1, step))
        list_load = [round(num, ndigit) for num in list_temp]
        list_load.reverse()
        
        ic.ILIM_TH = reg[0]
        ic.ILIM_TH_EX = reg[1]

        pre_iout = 0
        fist_op = True
        
        count = 0
        
        for n in range(1, 5):
            ss_v = voltage - r_cable * list_load[0] * 0.015 * n
            ld.vset = ss_v
            delay(0.2)
            print(ss_v)

        for set_load in list_load:
            
            ld.vset = set_load
            if fist_op:
                delay(0.5)
                fist_op = False
            
            off_by_lim = ic.OFF_BY_ILIM
            sw_sts = ic.SW_STS
            ilim_hys = ic.ILIM_HYS
            ilim_th = ic.ILIM_TH
            ilim_th_ex = ic.ILIM_TH_EX
            active_ilim_en = ic.ACTIVE_ILIM_EN
            
            for _ in range(3):
                iout = abs(round(dm2.current_10A, 6))
            # print(f"[{current:.03f}] {iout:.06f} ({sw_sts:#x}) {set_load}")
            print(ic.trm_isgn, ic.trm_isofs, voltage, off_by_lim, sw_sts, ilim_hys, ilim_th, ilim_th_ex, iout, current, count)
            
            if sw_sts != 2:
                count += 1
                if count > 15:
                    ld.vset = voltage
                    delay(1)
                    break
            else:
                if pre_iout < iout:
                    pre_iout = iout
                else:
                    pass
        
        ilim_acc = round(((pre_iout-current)/current*100), 6)
        ret = [ic.trm_isgn, ic.trm_isofs, voltage, pre_iout, ilim_th_ex, active_ilim_en, ilim_acc]
        log.output_csv(ret)
        
        print(f"trm_isgn : {ic.trm_isgn:#x} ({ic.trm_isgn})")
        print(f"trm_isofs : {ic.trm_isofs:#x}")
        
        ld.vset = voltage
        
        h.gpio_7(0)
        delay(0.5)

# ----------------------------------------------------------------------------

print(f"trm_isgn : {ic.trm_isgn:#x} ({ic.trm_isgn})")
print(f"trm_isofs : {ic.trm_isofs:#x}")
# disable_all()

In [ ]:
disable_all()